# Working with Scalar Fields

Now we are ready to combine a multidimensional `ParallelArray` with `Coordinate` objects to create a scalar field &mdash; a scalar-valued dataset defined over a multidimensional coordinate space.

## The ScalarField class

The `ScalarField` class encapsulates a `ParallelArray` containing the scalar data and a `CoordinateSpace` object holding the coordinates for each dimension.

To create a scalar field, you first need to create the parallel array holding the data values and each coordinate object, then pass them to the class constructor:

In [1]:
import numpy as np
import rockverse as rv

values = rv.array(np.random.rand(10, 15, 18), name='temperature')
x = rv.coordinate(0.5*np.arange(10, dtype=float), name='x')
y = rv.coordinate(0.75*np.arange(15, dtype=float), name='y')
z = rv.coordinate(2*np.arange(18, dtype=float), name='z')

s = rv.scalarfield(values, coords=(x, y, z))
print(s)


Arrays and coordinates can be accessed through their corresponding properties:

In [2]:
print(s.array)
print(s.array.name)
print(s.coords)
print(s.coords[0])
print(f"{s.coords[0].name}: {s.coords[0][...]}")

temperature
x: [0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  4.5]


Keep in mind that the array and the coordinates are passed by reference (shallow copies), so modifying one will be reflected in the other:

In [3]:
x[1] = 0.333 # Change made on the Coordinate object...
print(s.coords['x'][...]) # ... also appears in the scalar field

[0.    0.333 1.    1.5   2.    2.5   3.    3.5   4.    4.5  ]


In [4]:
print(values[0, 0, 0]) # Original array value
s.array[0, 0, 0] = 123 # Change made in the scalar field...
print(values[0, 0, 0]) # ... also appears in the original object

0.4419169372436801
123.0


## Sharing coordinates

The `coords` parameter can also be a full `CoordinateSpace` object. 
This makes it easy to create multiple scalar fields that share the same coordinates:

In [5]:
values2 = rv.array(np.random.rand(10, 15, 18), name='pressure')
s2 = rv.scalarfield(values2, coords=s.coords)
print(f"Sharing coordinate space? {s.coords is s2.coords}")

Sharing coordinate space? True


Again, remember that in this case, any change made to the original coordinates or the coordinate space in one scalar field will be reflected in all other scalar fields that share those objects.

In [6]:
print("Original")
print(f"z[3] = {z[3]}")
print(f"s.coords['z'][3] = {s.coords['z'][3]}")
print(f"s2.coords['z'][3] = {s2.coords['z'][3]}")

print("\nChanged")
s2.coords['z'][3] = 6.12345 # This will affect the others...
print(f"z[3] = {z[3]}")
print(f"s.coords['z'][3] = {s.coords['z'][3]}")
print(f"s2.coords['z'][3] = {s2.coords['z'][3]}")

Original
z[3] = 6.0
s.coords['z'][3] = 6.0
s2.coords['z'][3] = 6.0

Changed
z[3] = 6.12345
s.coords['z'][3] = 6.12345
s2.coords['z'][3] = 6.12345


## Auto-created coordinates

Finally, you can also omit the `coords` parameter. In this case, the coordinates will be automatically created as index-based coordinate arrays derived from the array’s shape:

In [10]:
s3 = rv.scalarfield(values)
print(s3.coords[0][...])
print(s3.coords[1][...])
print(s3.coords[2][...])

[0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14.]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.]


You can later modify the coordinates or data through their respective attributes, as demonstrated earlier.
Be sure to consult the [API documentation](../../../api/core/scalarfield.rst) for a comprehensive list of `ScalarField` attributes and methods.